# StatCore K-Means: robust, topology-aware and model-selecting variants

[Open in Colab](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/statcore-kmeans.ipynb)

**Paper topic:** Reliability-Oriented K-Means in StatCore — `docs/latex/statcore-kmeans.tex` ([repository](https://github.com/Jabualani/ML-Reliabilty/blob/main/docs/latex/statcore-kmeans.tex)).
**Experiment script reproduced here:** `docs/latex/experiments/kmeans_experiment.py` (reduced seed count so the notebook runs in a few minutes).

StatCore is a pure-NumPy/SciPy model family in `ml_reliability.statcore`. Every model has a `variant`
argument; each variant below is compared with the standard reference implementation.
**scikit-learn is used only as the baseline/reference** (and for loading public datasets), never inside StatCore.

## 1. Install (parameter cell)

> **Note:** StatCore has **not been pushed / published yet**. Until the repository is public, the
> `git+https` install below will fail. In that case use the **fallback cell**: zip the repository folder,
> upload the zip to Colab (left sidebar → Files, or the `files.upload()` helper) and `pip install` it.
> Run exactly one of the two cells.

In [ ]:
# @title Install StatCore (PARAMETER: change the source if needed)
# Option A - once the repository is published:
!pip install "git+https://github.com/Jabualani/ML-Reliabilty.git"

In [ ]:
# @title FALLBACK install from an uploaded zip of the repository (use if Option A fails)
# Zip the repo folder (ML-Reliabilty.zip), then uncomment and run:
# from google.colab import files
# files.upload()                       # pick ML-Reliabilty.zip
# !unzip -q -o ML-Reliabilty.zip -d repo
# !pip install ./repo/ML-Reliabilty     # adjust to the folder that contains pyproject.toml

In [ ]:
# @title Parameters and imports
SEEDS = (11, 23, 37)   # the paper uses more seeds; fewer here to keep the run short

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 2. Method summary

Data $x_i\in\mathbb{R}^d$, centres $c_1..c_k$. Lloyd iterations alternate assignment $z_i=\arg\min_j D(x_i,c_j)$ and centre updates; `n_init` restarts keep the lowest objective $\sum_i D(x_i,c_{z_i})$. Reference: sklearn `KMeans(n_init=5, max_iter=100, tol=1e-6)`.

**`base`** — squared Euclidean $D=\|x-c\|^2$, restarts seeded with $k$ distinct random data rows, mean update.

**`better_init`** — same Lloyd, seeded by k-means++: first centre uniform, then $P(x_i)\propto\min_j\|x_i-c_j\|^2$.

**`robust`** (trimmed coordinate-wise k-medians) — $D=\|x-c\|_1$; at each step the `trim` fraction (5 %) of points with the largest assignment cost is ignored, and each centre becomes the coordinate-wise **median** of its remaining members (random-row seeding as in `base`).

**`topology_aware`** — k-means++ seeding plus kNN-graph smoothing. With $N_m(i)$ the $m$ nearest neighbours (`n_neighbors`) and previous assignments $z^{\text{prev}}$,
$$\tilde D(x_i,c_j)=\frac{\|x_i-c_j\|^2}{\overline{\|\cdot\|^2}}-\lambda\cdot\frac{|\{l\in N_m(i): z^{\text{prev}}_l=j\}|}{m},\qquad \lambda=\texttt{topology\_weight}$$
(the smoothing term is only active from the second iteration on), so points prefer the cluster of their neighbours.

**`stat_model_selection`** — fits every $k\in[k_{\min},k_{\max}]$ and keeps the best by BIC or mean silhouette. For the spherical-Gaussian BIC with $\hat\sigma^2=\mathrm{SSE}/((n-k)d)$:
$$\log L=\sum_j n_j\log\frac{n_j}{n}-\frac{nd}{2}\log(2\pi\hat\sigma^2)-\frac{(n-k)d}{2},\qquad \mathrm{BIC}=-2\log L+(k-1+kd+1)\log n .$$
`scores_` stores BIC (lower is better) or silhouette (higher is better) per $k$.

## 3. Experiment (mirrors `kmeans_experiment.py`)

*synthetic*: three Gaussian blobs in 4-D ($\sigma=0.48$), 18 gross outliers injected into the training set only; *iris*: 75/25 split. Data are standardised with training statistics; the model is fitted on the training part and evaluated on the held-out part: adjusted Rand index (ARI) against the true labels, silhouette and held-out SSE (squared distance to the assigned centre). All methods get $k=3$ except `stat_model_selection`, which searches $k\in[2,6]$ by BIC.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans as ReferenceKMeans        # baseline only
from ml_reliability.statcore import KMeans
from ml_reliability.statcore.kmeans.kmeans import VARIANTS

METHODS = ("reference",) + tuple(VARIANTS)

def standardize(train, test):
    mean = train.mean(0); scale = train.std(0); scale[scale <= 1e-12] = 1.0
    return (train - mean) / scale, (test - mean) / scale

def synthetic(seed):
    rng = np.random.default_rng(seed)
    means = np.array([[-3, -2, 0, 1], [3, -2, 1, -1], [0, 3, -1, 0]])
    clean = np.vstack([rng.normal(loc=m, scale=.48, size=(100, 4)) for m in means]); labels = np.repeat(np.arange(3), 100)
    order = rng.permutation(len(clean)); tr, te = order[:240], order[240:]; train = clean[tr].copy()
    idx = rng.choice(len(train), 18, replace=False); train[idx] += rng.normal(0, 6, (18, 4))
    xtr, xte = standardize(train, clean[te]); return xtr, xte, labels[te], labels[tr]

def iris(seed):
    data = load_iris(); order = np.random.default_rng(seed).permutation(len(data.data)); split = int(.75 * len(order))
    tr, te = order[:split], order[split:]; xtr, xte = standardize(data.data[tr], data.data[te])
    return xtr, xte, data.target[te], data.target[tr]

def ari(a, b):
    _, ai = np.unique(a, return_inverse=True); _, bi = np.unique(b, return_inverse=True)
    tab = np.zeros((ai.max() + 1, bi.max() + 1), dtype=np.int64); np.add.at(tab, (ai, bi), 1)
    c2 = lambda x: x * (x - 1) / 2.0
    index = c2(tab).sum(); rows = c2(tab.sum(1)).sum(); cols = c2(tab.sum(0)).sum(); total = c2(len(a))
    expected = rows * cols / total if total else 0.; maximum = .5 * (rows + cols)
    return float((index - expected) / (maximum - expected)) if maximum != expected else 1.

def silhouette(X, labels):
    classes = np.unique(labels)
    if len(classes) < 2: return 0.
    D = np.sqrt(np.maximum((X ** 2).sum(1)[:, None] + (X ** 2).sum(1)[None, :] - 2 * X @ X.T, 0)); vals = np.zeros(len(X))
    for i in range(len(X)):
        own = labels == labels[i]; count = int(own.sum()); a = D[i, own].sum() / (count - 1) if count > 1 else 0.
        b = min(D[i, labels == c].mean() for c in classes if c != labels[i])
        vals[i] = (b - a) / max(a, b, 1e-12) if count > 1 else 0.
    return float(vals.mean())

def fit_method(method, xtr, k, seed):
    if method == "reference":
        return ReferenceKMeans(n_clusters=k, n_init=5, random_state=seed, max_iter=100, tol=1e-6).fit(xtr)
    kw = dict(variant=method, random_state=seed, max_iter=100, n_init=5, tol=1e-6)
    if method == "stat_model_selection":
        kw.update(k_range=(2, 6), criterion="bic")
    else:
        kw.update(n_clusters=k)
    return KMeans(**kw).fit(xtr)

def run(method, xtr, xte, k, seed):
    model = fit_method(method, xtr, k, seed)
    labels = model.predict(xte); centers = model.cluster_centers_
    return labels, float(np.sum((xte - centers[labels]) ** 2)), len(centers)

rows = []
for ds, factory in (("synthetic", synthetic), ("iris", iris)):
    for method in METHODS:
        for seed in SEEDS:
            xtr, xte, truth, _ = factory(seed)
            labels, sse, k_used = run(method, xtr, xte, 3, seed)
            rows.append(dict(dataset=ds, method=method, seed=seed, ARI=ari(truth, labels),
                             silhouette=silhouette(xte, labels), SSE=sse, k_used=k_used))
res = pd.DataFrame(rows)
mean = res.groupby(["dataset", "method"], sort=False)[["ARI", "silhouette", "SSE", "k_used"]].mean()
sd = res.groupby(["dataset", "method"], sort=False)[["ARI", "silhouette", "SSE"]].std()
print(f"Held-out metrics, mean over seeds {SEEDS}")
mean.round(4)

In [ ]:
print("sample SD over seeds"); print(sd.round(4).to_string())
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, metric in zip(axes, ("ARI", "silhouette", "SSE")):
    mean[metric].unstack("dataset").loc[list(METHODS)].plot.bar(ax=ax, rot=30); ax.set_title(f"{metric} (mean over seeds)")
plt.tight_layout(); plt.show()

## 4. Look inside each variant

Cluster assignments on the first two standardised features of the contaminated synthetic set, and the BIC / silhouette curves used by `stat_model_selection`.

In [ ]:
xtr, xte, truth, _ = synthetic(SEEDS[0])
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, m in zip(axes, ("reference", "base", "robust", "topology_aware")):
    model = fit_method(m, xtr, 3, SEEDS[0]); lab = model.predict(xte)
    ax.scatter(xte[:, 0], xte[:, 1], c=lab, s=10, cmap="viridis"); ax.scatter(*model.cluster_centers_[:, :2].T, c="red", marker="x", s=80)
    ax.set_title(f"{m}  ARI={ari(truth, lab):.2f}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, crit in zip(axes, ("bic", "silhouette")):
    sel = KMeans(variant="stat_model_selection", k_range=(2, 6), criterion=crit, random_state=SEEDS[0]).fit(xtr)
    ks = sorted(sel.scores_); ax.plot(ks, [sel.scores_[k] for k in ks], "o-"); ax.axvline(sel.n_clusters_, color="r", ls="--")
    ax.set_title(f"criterion={crit}: chosen k={sel.n_clusters_} (true k=3)"); ax.set_xlabel("k")
plt.tight_layout(); plt.show()

## 5. Summary and limitations

In [ ]:
t = mean.reset_index()
for metric, better in (("ARI", "max"), ("SSE", "min")):
    idx = t.groupby("dataset")[metric].transform(better) == t[metric]
    print(f"best {metric}:", dict(zip(t[idx]["dataset"], t[idx]["method"])))

**What to read from the tables.** In our local 3-seed run, `robust` was the best on ARI for both datasets, while `better_init` and `topology_aware` were *worse* than random-row seeding on the contaminated synthetic set (ARI ≈ 0.88 vs 1.0, large SD): $D^2$-weighted k-means++ seeding preferentially picks the injected outliers as centres. `stat_model_selection` chose $k\approx5$–$6$ (not 3) on both datasets, because BIC treats the outlier points as extra clusters (synthetic) and the overlapping iris classes need extra spherical components.

**Honest limitations**
- Two small datasets and a few seeds: ARI/SSE differences between several variants are often within one SD; iris clusters (versicolor/virginica) overlap, which caps every method.
- SSE (squared distance to centre) is *not* the objective of `robust` (L1), so reference-style SSE can favour non-robust methods even when ARI favours the robust one; compare metrics, do not rank on one.
- `topology_aware` adds a heuristic penalty (weights fixed at `topology_weight=0.5`, `n_neighbors=10`) and costs $O(n^2)$ for the neighbour graph; it can blur nearby clusters.
- BIC uses a spherical-Gaussian model, so it is misleading for elongated, overlapping or contaminated clusters (as seen above); `stat_model_selection` is not a reliable way to recover the true $k$ here, and its SSE is lower simply because it uses more centres.
- Silhouette is evaluated on the held-out part with held-out predictions only; no significance tests were run.